In [ ]:
%load_ext autoreload
%autoreload 2
import os
print(os.getcwd())
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader
import pickle
from utils import reproducibility
batch_size = 256
reproducibility(2025)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# simulation

In [ ]:
%load_ext autoreload
%autoreload 2
from data_process import generate_simulated_data

simudata_GTE, label_GTE = generate_simulated_data(sc_data="../result/expr/dat1_gtex_tissue_tpm_filter.txt",
                                   n=500, samplenum=5000, 
                                   d_prior=None, sparse=True)

simudata_HPA, label_HPA = generate_simulated_data(sc_data="../result/expr/dat1_hpa_tissue_tpm_filter.txt",
                                   n=500, samplenum=50, 
                                   d_prior=None, sparse=True)

print(simudata_GTE.shape)
print(label_GTE)
print(simudata_HPA.shape)
print(label_HPA)

In [ ]:
%load_ext autoreload
%autoreload 2
from data_process import GTEDataGeneVar, GTEDataGeneVarReal, GetXandYSelG, GetXandYSelGReal, PlotData, PlotDataTsne

variance_threshold = 0.9
all_genename = GTEDataGeneVar(simudata_GTE, vart = False)

GTE_x, GTE_y, celltypes = GetXandYSelG(simudata_GTE, all_genename, scaler="mms") 
HPA_x, HPA_y, celltypes = GetXandYSelG(simudata_HPA, all_genename, scaler="mms") 
print(GTE_x.shape)
print(GTE_y.shape)
print(HPA_x.shape)
print(HPA_y.shape)

real_pth = "../result/expr/all_exo_tpm_filter.txt"
real_x = GetXandYSelGReal(real_pth, all_genename, scaler="mms") 
print(real_x.shape)

# split data

In [ ]:
%load_ext autoreload
%autoreload 2
from data_process import DataSplitTrValTe

GTE_x_train, GTE_x_val, GTE_x_test, GTE_y_train, GTE_y_val, GTE_y_test = DataSplitTrValTe(GTE_x, GTE_y)

# Save data

In [ ]:
data_to_save = {

    'simudata_GTE': simudata_GTE,
    'label_GTE': list(label_GTE),
    'simudata_HPA': simudata_HPA,
    'label_HPA': list(label_HPA),

    'all_genename': all_genename,
    'celltypes': celltypes,

    'GTE_x': GTE_x,
    'GTE_y': GTE_y,
    'HPA_x': HPA_x,
    'HPA_y': HPA_y,
    'real_x': real_x,

    'GTE_x_train': GTE_x_train,
    'GTE_x_val': GTE_x_val,
    'GTE_x_test': GTE_x_test,
    'GTE_y_train': GTE_y_train,
    'GTE_y_val': GTE_y_val,
    'GTE_y_test': GTE_y_test,
}

with open(f'../result/data_abalation/Stim_data.pkl', 'wb') as file:
    pickle.dump(data_to_save, file)